# 📖 Notebook 2: Query Optimization

Even with perfect indexes, a badly written query can still be slow. In this notebook, we'll look at common query anti-patterns and how to fix them.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to read `EXPLAIN ANALYZE` output like a pro
- Why N+1 queries are a performance killer
- How to replace subqueries with JOINs
- When to use CTEs and window functions
- How to pick the right JOIN strategy

## The Pattern: BAD → BETTER → BEST

| Approach | Technique | Problem |
|----------|-----------|---------|
| 🔴 BAD | N+1 queries, SELECT *, correlated subqueries | Many round trips, excess data |
| 🟡 BETTER | JOINs, specific columns, simple aggregation | Single query, less data |
| 🟢 BEST | CTEs, window functions, proper JOINs | Optimal execution plan |

## 🛠️ Setup

```bash
cd deep-dives/postgres
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import time
from tabulate import tabulate

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "postgres_demo",
    "user": "demo",
    "password": "demo"
}

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

def run_query(sql, params=None, fetch=True):
    conn = get_conn()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql, params)
    result = cur.fetchall() if fetch else None
    cols = [desc[0] for desc in cur.description] if cur.description else []
    conn.close()
    return result, cols

def explain(sql, params=None):
    conn = get_conn()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(f"EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) {sql}", params)
    plan = cur.fetchall()
    conn.close()
    print("┌─── EXPLAIN ANALYZE ───────────────────────────────────")
    for row in plan:
        print(f"│ {row[0]}")
    print("└───────────────────────────────────────────────────────")

def timed_query(sql, params=None, label="Query"):
    times = []
    for _ in range(5):
        start = time.time()
        run_query(sql, params)
        times.append((time.time() - start) * 1000)
    avg = sum(times) / len(times)
    print(f"⏱️  {label}: {avg:.2f} ms (avg of 5 runs)")
    return avg

# Create indexes that a production DB would have
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
indexes = [
    "CREATE INDEX IF NOT EXISTS idx_posts_user_id ON posts(user_id)",
    "CREATE INDEX IF NOT EXISTS idx_comments_post_id ON comments(post_id)",
    "CREATE INDEX IF NOT EXISTS idx_comments_user_id ON comments(user_id)",
    "CREATE INDEX IF NOT EXISTS idx_likes_post_id ON likes(post_id)",
    "CREATE INDEX IF NOT EXISTS idx_likes_user_id ON likes(user_id)",
    "CREATE INDEX IF NOT EXISTS idx_dm_sender ON direct_messages(sender_id)",
    "CREATE INDEX IF NOT EXISTS idx_dm_receiver ON direct_messages(receiver_id)",
    "CREATE INDEX IF NOT EXISTS idx_dm_created ON direct_messages(created_at)",
]
for idx in indexes:
    cur.execute(idx)
cur.execute("ANALYZE")
conn.close()

print("✅ Connected and indexes created")

---

## 🔴 BAD: The N+1 Query Problem

The **N+1 problem** is the most common performance mistake in application code. It happens when you:

1. Run **1 query** to get a list of items
2. Run **N additional queries** — one for each item — to get related data

Example: "Show me the 20 most recent posts with their author names."

The **BAD** way does 21 database round trips (1 + 20). Each round trip adds network latency, connection overhead, and query planning time.

In [ ]:
# 🔴 BAD: The N+1 pattern
# Step 1: Get the 20 most recent posts
# Step 2: For EACH post, make a separate query to get the author name

print("=" * 60)
print("🔴 BAD: N+1 queries — 21 round trips to the database!")
print("=" * 60)
print()

start = time.time()

# Query 1: Get 20 recent posts
conn = get_conn()
cur = conn.cursor()
cur.execute("SELECT id, user_id, title, created_at FROM posts ORDER BY created_at DESC LIMIT 20")
posts = cur.fetchall()
conn.close()

# Queries 2-21: Get each author's name individually (BAD!)
results = []
for post_id, user_id, title, created_at in posts:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SELECT username FROM users WHERE id = %s", (user_id,))
    author = cur.fetchone()[0]
    conn.close()
    results.append((title[:40], author, str(created_at)[:19]))

bad_n1_time = (time.time() - start) * 1000

print(tabulate(results[:5], headers=["Title", "Author", "Created"], tablefmt="simple_grid"))
print(f"  ... and 15 more rows")
print()
print(f"⏱️  Total time: {bad_n1_time:.2f} ms")
print(f"📡 Database round trips: 21  (1 + 20)")
print()
print("💡 Each round trip costs ~1-5ms of overhead.")
print("   With 1000 posts, that's 1001 queries!")

## 🟡 BETTER: Use a JOIN

Instead of N+1 queries, use a **single JOIN** query. PostgreSQL fetches all the data in one round trip and combines the tables internally — much faster than doing it in application code.

In [ ]:
# 🟡 BETTER: Single JOIN query — 1 round trip instead of 21

print("=" * 60)
print("🟡 BETTER: Single JOIN — 1 round trip")
print("=" * 60)
print()

join_query = """
    SELECT p.title, u.username AS author, p.created_at
    FROM posts p
    JOIN users u ON u.id = p.user_id
    ORDER BY p.created_at DESC
    LIMIT 20
"""

start = time.time()
results, cols = run_query(join_query)
better_time = (time.time() - start) * 1000

display = [(r[0][:40], r[1], str(r[2])[:19]) for r in results[:5]]
print(tabulate(display, headers=["Title", "Author", "Created"], tablefmt="simple_grid"))
print(f"  ... and 15 more rows")
print()
print(f"⏱️  Total time: {better_time:.2f} ms")
print(f"📡 Database round trips: 1")
print(f"🚀 Speedup: {bad_n1_time / better_time:.1f}× faster!")
print()

explain(join_query)

---

## 🔴 BAD: SELECT * and Unnecessary Data

Another common mistake is `SELECT *` — fetching **all columns** when you only need a few. This wastes:
- **Network bandwidth**: sending data you don't need
- **Memory**: loading unused columns into your app
- **I/O**: reading wider rows from disk

In [ ]:
# 🔴 BAD: SELECT * fetches ALL columns including big TEXT fields

print("=" * 60)
print("🔴 BAD: SELECT * — fetches everything, including large TEXT columns")
print("=" * 60)
print()

bad_select = "SELECT * FROM posts WHERE user_id = 42"
better_select = "SELECT id, title, like_count, created_at FROM posts WHERE user_id = 42"

bad_time = timed_query(bad_select, label="SELECT * (all columns)")
better_time = timed_query(better_select, label="SELECT specific columns")

print()
print(f"🚀 Selecting specific columns: {bad_time / better_time:.1f}× faster")
print()

# Show the data size difference
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
cur.execute("SELECT pg_column_size(t.*) FROM posts t WHERE user_id = 42 LIMIT 1")
full_size = cur.fetchone()[0]
cur.execute("SELECT pg_column_size(ROW(id, title, like_count, created_at)) FROM posts WHERE user_id = 42 LIMIT 1")
partial_size = cur.fetchone()[0]
conn.close()

print(f"📏 Row size with SELECT *:        {full_size} bytes")
print(f"📏 Row size with specific columns: {partial_size} bytes")
print(f"💡 That's {full_size / partial_size:.1f}× less data per row!")

---

## 🔴 BAD: Correlated Subqueries

A **correlated subquery** runs once for every row in the outer query. It's like the N+1 problem, but inside SQL itself.

In [ ]:
# 🔴 BAD: Correlated subquery — runs the subquery for EACH row

print("=" * 60)
print("🔴 BAD: Correlated subquery (subquery runs per row)")
print("=" * 60)
print()

bad_subquery = """
    SELECT u.username,
           (SELECT COUNT(*) FROM posts WHERE user_id = u.id) AS post_count,
           (SELECT COUNT(*) FROM comments WHERE user_id = u.id) AS comment_count
    FROM users u
    WHERE u.is_verified = TRUE
"""

explain(bad_subquery)
print()
bad_sub_time = timed_query(bad_subquery, label="Correlated subqueries")

## 🟡 BETTER: Replace with JOINs and GROUP BY

In [ ]:
# 🟡 BETTER: Use LEFT JOINs with GROUP BY

print("=" * 60)
print("🟡 BETTER: LEFT JOIN + GROUP BY")
print("=" * 60)
print()

better_join = """
    SELECT u.username,
           COUNT(DISTINCT p.id) AS post_count,
           COUNT(DISTINCT c.id) AS comment_count
    FROM users u
    LEFT JOIN posts p ON p.user_id = u.id
    LEFT JOIN comments c ON c.user_id = u.id
    WHERE u.is_verified = TRUE
    GROUP BY u.id, u.username
"""

explain(better_join)
print()
better_join_time = timed_query(better_join, label="LEFT JOIN + GROUP BY")
print()
print(f"🚀 Speedup: {bad_sub_time / better_join_time:.1f}× faster")

## 🟢 BEST: Lateral Joins or Pre-aggregated Subqueries

When JOINs cause row explosion (many-to-many), pre-aggregate in subqueries first, then join.

In [ ]:
# 🟢 BEST: Pre-aggregated subqueries — aggregate BEFORE joining

print("=" * 60)
print("🟢 BEST: Pre-aggregated subqueries (no row explosion)")
print("=" * 60)
print()

best_query = """
    SELECT u.username,
           COALESCE(pc.cnt, 0) AS post_count,
           COALESCE(cc.cnt, 0) AS comment_count
    FROM users u
    LEFT JOIN (
        SELECT user_id, COUNT(*) AS cnt
        FROM posts
        GROUP BY user_id
    ) pc ON pc.user_id = u.id
    LEFT JOIN (
        SELECT user_id, COUNT(*) AS cnt
        FROM comments
        GROUP BY user_id
    ) cc ON cc.user_id = u.id
    WHERE u.is_verified = TRUE
"""

explain(best_query)
print()
best_time = timed_query(best_query, label="Pre-aggregated subqueries")
print()

print("📊 Comparison:")
print(f"  🔴 Correlated subqueries: {bad_sub_time:>8.2f} ms")
print(f"  🟡 LEFT JOIN + GROUP BY:  {better_join_time:>8.2f} ms")
print(f"  🟢 Pre-aggregated:        {best_time:>8.2f} ms")

---

## 🟢 BEST: Window Functions

Window functions let you compute aggregates **without collapsing rows**. They're perfect for ranking, running totals, and comparisons within groups.

In [ ]:
# Task: For each user, find their most popular post (highest like_count)

# 🔴 BAD: Correlated subquery approach
bad_window = """
    SELECT p.user_id, p.title, p.like_count
    FROM posts p
    WHERE p.like_count = (
        SELECT MAX(like_count) FROM posts WHERE user_id = p.user_id
    )
    AND p.user_id <= 100
"""

# 🟢 BEST: Window function approach
best_window = """
    SELECT user_id, title, like_count
    FROM (
        SELECT user_id, title, like_count,
               ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY like_count DESC) AS rn
        FROM posts
        WHERE user_id <= 100
    ) ranked
    WHERE rn = 1
"""

print("=" * 60)
print("Task: Find each user's most popular post")
print("=" * 60)
print()

bad_w_time = timed_query(bad_window, label="🔴 BAD: Correlated subquery")
best_w_time = timed_query(best_window, label="🟢 BEST: Window function (ROW_NUMBER)")
print()
print(f"🚀 Window function: {bad_w_time / best_w_time:.1f}× faster")
print()

print("--- Window function EXPLAIN ---")
explain(best_window)
print()
print("💡 Window functions are powerful for:")
print("   - ROW_NUMBER(): pick the top-N per group")
print("   - RANK() / DENSE_RANK(): handle ties")
print("   - LAG() / LEAD(): compare with previous/next row")
print("   - SUM() OVER: running totals")

## 🧹 Cleanup

In [ ]:
# Clean up indexes
conn = get_conn()
conn.autocommit = True
cur = conn.cursor()
for idx in ['idx_posts_user_id', 'idx_comments_post_id', 'idx_comments_user_id',
            'idx_likes_post_id', 'idx_likes_user_id',
            'idx_dm_sender', 'idx_dm_receiver', 'idx_dm_created']:
    cur.execute(f"DROP INDEX IF EXISTS {idx}")
conn.close()
print("🧹 Indexes removed")

## 📚 Summary

### Key Takeaways

1. **N+1 queries are the #1 performance killer** — always use JOINs instead of loops
2. **SELECT only the columns you need** — `SELECT *` wastes bandwidth and memory
3. **Correlated subqueries are slow** — pre-aggregate or use window functions
4. **Window functions are your best friend** for ranking, top-N, and analytics
5. **Always check EXPLAIN ANALYZE** before and after optimizing

### Query Optimization Cheat Sheet

| Bad Pattern | Good Replacement |
|------------|-----------------|
| N+1 queries (loop) | Single JOIN query |
| `SELECT *` | `SELECT col1, col2` |
| Correlated subquery | JOIN + GROUP BY or pre-aggregate |
| Multiple separate queries | CTE (WITH clause) |
| Self-join for ranking | Window function |

### Next Up

In **Notebook 3**, we'll explore **replication and failover** — how to scale reads and survive server failures.